# LC 121 — Best Time to Buy and Sell Stock
**Difficulty:** Easy &nbsp;|&nbsp; **Category:** Sliding Window
**Pattern:** One-Pass Running Minimum

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Track the lowest price
seen so far as you scan left to right. At each price,
the best profit you could make selling today is
current price minus that running minimum.
</div>

## Official Problem Statement

You are given an array `prices` where `prices[i]`
is the price of a given stock on the `i`th day.

You want to maximize your profit by choosing a
single day to buy one stock and choosing a different
day in the future to sell that stock.

Return the maximum profit you can achieve from this
transaction. If you cannot achieve any profit,
return `0`.

**Example 1:**
```
Input:  prices = [7,1,5,3,6,4]
Output: 5
Explanation: Buy on day 2 (price=1), sell on day 5
             (price=6), profit = 6-1 = 5.
```
**Example 2:**
```
Input:  prices = [7,6,4,3,1]
Output: 0
Explanation: No profitable transaction possible.
```

**Constraints:**
- `1 <= prices.length <= 10^5`
- `0 <= prices[i] <= 10^4`

## What This Is Actually Asking

Pick one day to buy and a later day to sell.
You must buy before you sell — no time travel.
Find the single buy-sell pair that gives the biggest
profit.
If every later price is lower than the buy price,
return 0 (do nothing).

## Walk Through an Example by Hand

```
prices = [7, 1, 5, 3, 6, 4]
          0  1  2  3  4  5

min_price = inf   max_profit = 0

day 0  price=7   7 < inf -> min_price=7
                 profit = 7-7 = 0   max_profit=0

day 1  price=1   1 < 7   -> min_price=1
                 profit = 1-1 = 0   max_profit=0

day 2  price=5   5 > 1   -> min stays 1
                 profit = 5-1 = 4   max_profit=4

day 3  price=3   3 > 1   -> min stays 1
                 profit = 3-1 = 2   max_profit=4

day 4  price=6   6 > 1   -> min stays 1
                 profit = 6-1 = 5   max_profit=5  <-

day 5  price=4   4 > 1   -> min stays 1
                 profit = 4-1 = 3   max_profit=5

Answer: 5
```

## The Picture

```
prices = [7, 1, 5, 3, 6, 4]

  7 |*
  6 |          *
  5 |     *
  4 |               *
  3 |          *
  2 |
  1 |     *      <- buy here (running minimum)
     day: 0  1  2  3  4  5

At every point, the best sell is today's price.
The best buy is the lowest price BEFORE today.
Just track the running minimum.

  current  |  min so far  |  profit today  |  best
  ---------|--------------|-----------     |-----
    7       |     7        |     0          |   0
    1       |     1        |     0          |   0
    5       |     1        |     4          |   4
    3       |     1        |     2          |   4
    6       |     1        |     5          |   5  <-
    4       |     1        |     3          |   5
```

## When To Use This Pattern

- When you see **max gain = later value minus earlier
  value**, think **track running minimum**
- When you must buy before you sell (order matters),
  think **one left-to-right pass — no look-ahead**
- When the answer is zero if no profit exists, think
  **initialise max_profit = 0**
- When extended to multiple transactions, think
  **DP or greedy sum of all positive day-to-day deltas**

## The Approach

Walk through the price list once, keeping track of
two things: the lowest price seen so far and the
best profit seen so far.
At each price, if it is lower than the running
minimum, update the minimum.
Otherwise compute the profit if you sold today and
update the best profit if it is larger.
Return the best profit at the end.

In [1]:
from typing import List  # type hints for the solution

In [2]:
def test_harness(func):
    tests = [
        # (prices, expected)
        ([7,1,5,3,6,4],  5),
        ([7,6,4,3,1],    0),   # descending — no profit
        ([1,2],          1),   # two days, profit
        ([2,1],          0),   # two days, no profit
        ([1],            0),   # single day
        ([3,3,3],        0),   # flat — no profit
        ([1,10,2,9],     9),   # buy low, sell highest
        ([2,4,1,7],      6),   # buy after initial peak
        ([0,10000],      10000),  # max spread
    ]

    passed = 0
    for i, (prices, expected) in enumerate(tests):
        result = func(prices[:])
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"prices={prices} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def maxProfit(prices: List[int]) -> int:
    """
    Return maximum profit from one buy and one sell.

    One pass: track min_price seen so far and
    max_profit seen so far. At each price, update
    min_price if lower, else compute profit today
    and update max_profit. Return max_profit.

    Time:  O(n) — single pass through prices
    Space: O(1) — two variables only
    """
    pass


# Quick debug — run this cell while building
print(maxProfit([7,1,5,3,6,4]))  # 5
print(maxProfit([7,6,4,3,1]))    # 0
print(maxProfit([1,2]))           # 1
print(maxProfit([2,4,1,7]))      # 6

In [ ]:
# Uncomment and run when solution is ready
# test_harness(maxProfit)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — try every pair | O(n²) | O(1) |
| One-pass running minimum | O(n) | O(1) |

The running minimum eliminates the inner loop —
you never need to look backward because the minimum
is already being tracked.

## Real World Connection

At Citi, capacity planning uses the exact same
pattern to find the maximum CPU spike above a
rolling baseline: the "buy" is the baseline and
the "sell" is the peak, and we want the largest
deviation across the 90-day window.
The running-minimum scan runs over millions of
telemetry rows in a single Athena pass — fast
enough to run hourly before the Prophet forecasting
pipeline picks up the output.
On AWS, Lambda functions monitoring spot-instance
pricing use the same one-pass scan to flag the
optimal buy window before the next auction cycle.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra

In [7]:
import math
def maxProfit(prices: List[int]) -> int:
    """
    Return maximum profit from one buy and one sell.

    One pass: track min_price seen so far and
    max_profit seen so far. At each price, update
    min_price if lower, else compute profit today
    and update max_profit. Return max_profit.

    Time:  O(n) — single pass through prices
    Space: O(1) — two variables only
    """
    buyPrice,profit = math.inf ,0

    for p in prices:
        if p < buyPrice:
            buyPrice = p
            continue                    #do not buy and sell on the same price
        profit = max(profit, (p - buyPrice))
    return profit
        
r"""
5
0
1
6
Test 1: PASSED | prices=[7, 1, 5, 3, 6, 4] | expected=5 | got=5
Test 2: PASSED | prices=[7, 6, 4, 3, 1] | expected=0 | got=0
Test 3: PASSED | prices=[1, 2] | expected=1 | got=1
Test 4: PASSED | prices=[2, 1] | expected=0 | got=0
Test 5: PASSED | prices=[1] | expected=0 | got=0
Test 6: PASSED | prices=[3, 3, 3] | expected=0 | got=0
Test 7: PASSED | prices=[1, 10, 2, 9] | expected=9 | got=9
Test 8: PASSED | prices=[2, 4, 1, 7] | expected=6 | got=6
Test 9: PASSED | prices=[0, 10000] | expected=10000 | got=10000

9/9 tests passed

"""


# Quick debug — run this cell while building
print(maxProfit([7,1,5,3,6,4]))  # 5
print(maxProfit([7,6,4,3,1]))    # 0
print(maxProfit([1,2]))           # 1
print(maxProfit([2,4,1,7]))      # 6
test_harness(maxProfit)

5
0
1
6
Test 1: PASSED | prices=[7, 1, 5, 3, 6, 4] | expected=5 | got=5
Test 2: PASSED | prices=[7, 6, 4, 3, 1] | expected=0 | got=0
Test 3: PASSED | prices=[1, 2] | expected=1 | got=1
Test 4: PASSED | prices=[2, 1] | expected=0 | got=0
Test 5: PASSED | prices=[1] | expected=0 | got=0
Test 6: PASSED | prices=[3, 3, 3] | expected=0 | got=0
Test 7: PASSED | prices=[1, 10, 2, 9] | expected=9 | got=9
Test 8: PASSED | prices=[2, 4, 1, 7] | expected=6 | got=6
Test 9: PASSED | prices=[0, 10000] | expected=10000 | got=10000

9/9 tests passed
